In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc gem-suite matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Anggaran penggunaan: 3 minit pada pemproses Heron r2 (NOTA: Ini adalah anggaran sahaja. Masa jalan anda mungkin berbeza.)*

## Latar belakang

Tutorial ini menunjukkan cara merealisasikan peralihan fasa Nishimori pada pemproses kuantum IBM&reg;. Eksperimen ini pada asalnya diterangkan dalam [*Realizing the Nishimori transition across the error threshold for constant-depth quantum circuits*](https://arxiv.org/abs/2309.02863).

Peralihan fasa Nishimori merujuk kepada peralihan antara fasa teratur jarak pendek dan jarak jauh dalam model Ising ikatan rawak. Pada komputer kuantum, fasa teratur jarak jauh terzahir sebagai keadaan di mana Qubit-Qubit saling terbelit merentasi keseluruhan peranti. Keadaan yang sangat terbelit ini disediakan menggunakan protokol *generation of entanglement by measurement* (GEM). Dengan memanfaatkan pengukuran mid-Circuit, protokol GEM mampu membelit Qubit-Qubit merentasi keseluruhan peranti menggunakan Circuit yang hanya berkedalaman malar. Tutorial ini menggunakan pelaksanaan protokol GEM daripada pakej perisian [GEM Suite](https://github.com/qiskit-community/gem-suite).

## Keperluan

Sebelum memulakan tutorial ini, pastikan anda telah memasang perkara berikut:

- Qiskit SDK v1.0 atau lebih baru, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 atau lebih baru ( `pip install qiskit-ibm-runtime` )
- GEM Suite ( `pip install gem-suite` )

## Persediaan

In [2]:
import matplotlib.pyplot as plt

from collections import defaultdict

from qiskit_ibm_runtime import QiskitRuntimeService

from qiskit.transpiler import generate_preset_pass_manager

from gem_suite import PlaquetteLattice
from gem_suite.experiments import GemExperiment

## Langkah 1: Petakan input klasik kepada masalah kuantum
Protokol GEM beroperasi pada pemproses kuantum dengan kesambungan Qubit yang digambarkan oleh kekisi. Pemproses kuantum IBM hari ini menggunakan [kekisi hex berat](https://www.ibm.com/quantum/blog/heavy-hex-lattice). Qubit-Qubit pada pemproses dikumpulkan ke dalam *plaquette* berdasarkan sel unit kekisi yang mereka duduki. Oleh kerana satu Qubit mungkin terdapat dalam lebih daripada satu sel unit, plaquette-plaquette ini tidak disjoin. Pada kekisi hex berat, satu plaquette mengandungi 12 Qubit. Plaquette-plaquette itu sendiri juga membentuk kekisi, di mana dua plaquette disambungkan jika mereka berkongsi Qubit. Pada kekisi hex berat, plaquette-plaquette jiran berkongsi 3 Qubit.

Dalam pakej perisian GEM Suite, kelas asas untuk melaksanakan protokol GEM ialah `PlaquetteLattice`, yang mewakili kekisi plaquette (yang berbeza daripada kekisi hex berat). `PlaquetteLattice` boleh dimulakan daripada peta gandinggan Qubit. Buat masa ini, hanya peta gandinggan hex berat yang disokong.

Sel kod berikut memulakan kekisi plaquette daripada peta gandinggan pemproses kuantum IBM. Kekisi plaquette tidak semestinya merangkumi keseluruhan perkakasan. Sebagai contoh, `ibm_torino` mempunyai 133 Qubit secara keseluruhan, tetapi kekisi plaquette terbesar yang muat pada peranti itu hanya menggunakan 125 daripadanya, dan merangkumi 18 plaquette secara keseluruhan. Perkara serupa boleh diperhatikan pada peranti IBM Quantum&reg; dengan bilangan Qubit yang berbeza juga.

In [ ]:
# QiskitRuntimeService.save_account(channel="ibm_quantum", token="<YOUR_API_KEYN>", overwrite=True, set_as_default=True)
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
plaquette_lattice = PlaquetteLattice.from_coupling_map(backend.coupling_map)

print(f"Number of qubits in backend: {backend.num_qubits}")
print(
    f"Number of qubits in plaquette lattice: {len(list(plaquette_lattice.qubits()))}"
)
print(f"Number of plaquettes: {len(list(plaquette_lattice.plaquettes()))}")

Number of qubits in backend: 133
Number of qubits in plaquette lattice: 125
Number of plaquettes: 18


You can visualize the plaquette lattice by generating a diagram of its graph representation. In the diagram, the plaquettes are represented by labeled hexagons, and two plaquettes are connected by an edge if they share qubits.

In [7]:
plaquette_lattice.draw_plaquettes()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif" alt="Output of the previous code cell" />

Anda boleh menggambarkan kekisi plaquette dengan menjana rajah representasi grafnya. Dalam rajah tersebut, plaquette-plaquette diwakili oleh heksagon berlabel, dan dua plaquette disambungkan oleh tepi jika mereka berkongsi Qubit.

In [8]:
# Get a list of the plaquettes
plaquettes = list(plaquette_lattice.plaquettes())
# Display information about plaquette 0
plaquettes[0]

PyPlaquette(index=0, qubits=[0, 1, 2, 3, 4, 15, 16, 19, 20, 21, 22, 23], neighbors=[3, 1])

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/625882a4-faeb-4d96-b441-c989f43c4dea-0.avif)

Anda boleh mendapatkan maklumat tentang plaquette individu, seperti Qubit yang dikandunginya, menggunakan kaedah `plaquettes`.

In [9]:
plaquette_lattice.draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif" alt="Output of the previous code cell" />

In addition to the qubit labels and the edges indicating which qubits are connected, the diagram contains three additional pieces of information that are relevant to the GEM protocol:
- Each qubit is either shaded (gray) or unshaded. The shaded qubits are "site" qubits that represent the sites of the Ising model, and the unshaded qubits are "bond" qubits used to mediate interactions between the site qubits.
- Each site qubit is labeled either (A) or (B), indicating one of two roles a site qubit can play in the GEM protocol (the roles are explained later).
- Each edge is colored using one of six colors, thus partitioning the edges into six groups. This partitioning determines how two-qubit gates can be parallelized, as well as different scheduling patterns that are likely to incur different amounts of error on a noisy quantum processor. Because edges in a group are disjoint, a layer of two-qubit gates can be applied on those edges simultaneously. In fact, it is possible to partition the six colors into three groups of two colors such that the union of each group of two colors is still disjoint. Therefore, only three layers of two-qubit gates are needed to activate every edge. There are 12 ways to so partition the six colors, and each such partition yields a different 3-layer gate schedule.

Now that you have created a plaquette lattice, the next step is to initialize a `GemExperiment` object, passing both the plaquette lattice and the backend that you intend to run the experiment on. The `GemExperiment` class manages the actual implementation of the GEM protocol, including generating circuits, submitting jobs, and analyzing the data. The following code cell initializes the experiment class while restricting the plaquette lattice to only two of the plaquettes (21 qubits), reducing the size of the experiment to ensure that the noise in the hardware doesn't overwhelm the signal.

In [16]:
gem_exp = GemExperiment(plaquette_lattice.filter([9, 12]), backend=backend)

# visualize the plaquette lattice after filtering
plaquette_lattice.filter([9, 12]).draw_qubits()

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif" alt="Output of the previous code cell" />

Anda juga boleh menghasilkan rajah Qubit-Qubit asas yang membentuk kekisi plaquette.

In [12]:
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

Total number of circuits: 252


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/a19d63ce-3572-4081-a008-c1332fbbe303-0.avif)

Selain label Qubit dan tepi yang menunjukkan Qubit mana yang disambungkan, rajah ini mengandungi tiga maklumat tambahan yang berkaitan dengan protokol GEM:
- Setiap Qubit sama ada berlorek (kelabu) atau tidak berlorek. Qubit berlorek adalah Qubit "tapak" yang mewakili tapak model Ising, manakala Qubit tidak berlorek adalah Qubit "ikatan" yang digunakan untuk menjadi perantara interaksi antara Qubit tapak.
- Setiap Qubit tapak dilabel sama ada (A) atau (B), menunjukkan salah satu daripada dua peranan yang boleh dimainkan oleh Qubit tapak dalam protokol GEM (peranan-peranan itu dijelaskan kemudian).
- Setiap tepi diwarnakan menggunakan salah satu daripada enam warna, seterusnya membahagikan tepi kepada enam kumpulan. Pembahagian ini menentukan cara Gate dua-Qubit boleh diparalelkan, serta corak penjadualan berbeza yang mungkin menanggung jumlah ralat yang berbeza pada pemproses kuantum berbising. Oleh kerana tepi dalam satu kumpulan adalah disjoin, satu lapisan Gate dua-Qubit boleh digunakan pada tepi tersebut secara serentak. Malah, adalah mungkin untuk membahagikan enam warna ke dalam tiga kumpulan dua warna sedemikian rupa sehingga gabungan setiap kumpulan dua warna masih disjoin. Oleh itu, hanya tiga lapisan Gate dua-Qubit diperlukan untuk mengaktifkan setiap tepi. Terdapat 12 cara untuk membahagikan enam warna sedemikian, dan setiap pembahagian menghasilkan jadual Gate 3-lapisan yang berbeza.

Kini setelah anda mencipta kekisi plaquette, langkah seterusnya ialah memulakan objek `GemExperiment`, dengan menghantar kekisi plaquette dan Backend yang anda ingin jalankan eksperimen tersebut. Kelas `GemExperiment` menguruskan pelaksanaan sebenar protokol GEM, termasuk menjana Circuit, menghantar kerja, dan menganalisis data. Sel kod berikut memulakan kelas eksperimen sambil mengehadkan kekisi plaquette kepada hanya dua plaquette (21 Qubit), mengurangkan saiz eksperimen untuk memastikan hingar dalam perkakasan tidak menenggelami isyarat.

In [13]:
# Restrict experiment to the first scheduling pattern
gem_exp.set_experiment_options(schedule_idx=0)

# There are less circuits now
circuits = gem_exp.circuits()
print(f"Total number of circuits: {len(circuits)}")

# Print the RZZ angles swept over
print(f"RZZ angles:\n{gem_exp.parameters()}")

Total number of circuits: 21
RZZ angles:
[0.         0.07853982 0.15707963 0.23561945 0.31415927 0.39269908
 0.4712389  0.54977871 0.62831853 0.70685835 0.78539816 0.86393798
 0.9424778  1.02101761 1.09955743 1.17809725 1.25663706 1.33517688
 1.41371669 1.49225651 1.57079633]


![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/02357c6e-5c83-4ac0-811d-22602d9f33d5-0.avif)

Circuit protokol GEM dibina menggunakan langkah-langkah berikut:
1. Sediakan keadaan semua-$|+\rangle$ dengan menggunakan Gate Hadamard pada setiap Qubit.
2. Gunakan Gate $R_{ZZ}$ antara setiap pasangan Qubit yang disambungkan. Ini boleh dicapai menggunakan 3 lapisan Gate. Setiap Gate $R_{ZZ}$ bertindak pada Qubit tapak dan Qubit ikatan. Jika Qubit tapak dilabel (B), maka sudutnya ditetapkan pada $\frac{\pi}{2}$. Jika Qubit tapak dilabel (A), maka sudutnya dibenarkan untuk berubah, menghasilkan Circuit yang berbeza. Secara lalai, julat sudut ditetapkan kepada 21 titik berjarak sama antara $0$ dan $\frac{\pi}{2}$, inklusif.
3. Ukur setiap Qubit ikatan dalam asas Pauli $X$. Oleh kerana Qubit diukur dalam asas Pauli $Z$, ini boleh dicapai dengan menggunakan Gate Hadamard sebelum mengukur Qubit.

Perhatikan bahawa kertas kerja yang disebut dalam pengenalan tutorial ini menggunakan konvensyen berbeza untuk sudut $R_{ZZ}$, yang berbeza daripada konvensyen yang digunakan dalam tutorial ini sebanyak faktor 2.

Dalam langkah 3, hanya Qubit ikatan yang diukur. Untuk memahami keadaan yang tinggal pada Qubit tapak, adalah berguna untuk mempertimbangkan kes di mana sudut $R_{ZZ}$ yang digunakan pada Qubit tapak (A) dalam langkah 2 adalah sama dengan $\frac{\pi}{2}$. Dalam kes ini, Qubit tapak ditinggalkan dalam keadaan yang sangat terbelit yang serupa dengan keadaan GHZ,

$$
\lvert \text{GHZ} \rangle = \lvert 00 \cdots 00 \rangle + \lvert 11 \cdots 11 \rangle.
$$

Disebabkan kerawakan dalam hasil pengukuran, keadaan sebenar Qubit tapak mungkin adalah keadaan lain dengan perintah jarak jauh, contohnya, $\lvert 00110 \rangle + \lvert 11001 \rangle$. Walau bagaimanapun, keadaan GHZ boleh dipulihkan dengan menggunakan operasi penyahkodan berdasarkan hasil pengukuran. Apabila sudut $R_{ZZ}$ dikurangkan daripada $\frac{\pi}{2}$, perintah jarak jauh masih boleh dipulihkan sehingga sudut kritikal, yang dalam ketiadaan hingar, adalah kira-kira $0.3 \pi$. Di bawah sudut ini, keadaan yang terhasil tidak lagi mempamerkan belitan jarak jauh. Peralihan antara kehadiran dan ketiadaan perintah jarak jauh ini adalah peralihan fasa Nishimori.

Dalam huraian di atas, Qubit tapak dibiarkan tidak diukur, dan operasi penyahkodan boleh dilakukan dengan menggunakan Gate kuantum. Dalam eksperimen seperti yang dilaksanakan dalam GEM suite, yang diikuti oleh tutorial ini, Qubit tapak sebenarnya diukur, dan operasi penyahkodan dilakukan dalam langkah pasca-pemprosesan klasik.

Dalam huraian di atas, operasi penyahkodan boleh dilakukan dengan menggunakan Gate kuantum pada Qubit tapak untuk memulihkan keadaan kuantum. Walau bagaimanapun, jika tujuannya adalah untuk mengukur keadaan dengan segera, contohnya, untuk tujuan pencirian, maka Qubit tapak diukur bersama dengan Qubit ikatan, dan operasi penyahkodan boleh dilakukan dalam langkah pasca-pemprosesan klasik. Inilah cara eksperimen dilaksanakan dalam GEM suite, yang diikuti oleh tutorial ini.

Selain bergantung pada sudut $R_{ZZ}$ dalam langkah 2, yang secara lalai menyapu merentasi 21 nilai, Circuit protokol GEM juga bergantung pada corak penjadualan yang digunakan untuk melaksanakan 3 lapisan Gate $R_{ZZ}$. Seperti yang dibincangkan sebelum ini, terdapat 12 corak penjadualan sedemikian. Oleh itu, jumlah bilangan Circuit dalam eksperimen ialah $21 \times 12 = 252$.

Circuit-Circuit eksperimen boleh dijana menggunakan kaedah `circuits` kelas `GemExperiment`.

In [14]:
# Get the circuit at index 5
circuit = circuits[5]
# Remove the final measurements to ease visualization
circuit.remove_final_measurements()
# Draw the circuit
circuit.draw("mpl", fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif" alt="Output of the previous code cell" />

## Step 2: Optimize problem for quantum hardware execution

Transpiling quantum circuits for execution on hardware typically involves a [number of stages](/docs/guides/transpiler-stages). Typically, the stages that incur the most computational overhead are choosing the qubit layout, routing the two-qubit gates to conform to the qubit connectivity of the hardware, and optimizing the circuit to minimize its gate count and depth. In the GEM protocol, the layout and routing stages are unnecessary because the hardware connectivity is already incorporated into the design of the protocol. The circuits already have a qubit layout, and the two-qubit gates are already mapped onto native connections. Furthermore, in order to preserve the structure of the circuit as the $R_{ZZ}$ angle is varied, only very basic circuit optimization should be performed.

The `GemExperiment` class transparently transpiles circuits when executing the experiment. The layout and routing stages are already overridden by default to do nothing, and circuit optimization is performed at a level that only optimizes single-qubit gates. However, you can override or pass additional options using the `set_transpile_options` method. For the sake of visualization, the following code cell manually transpiles the circuit displayed previously, and draws the transpiled circuit.

In [15]:
# Demonstrate setting transpile options
gem_exp.set_transpile_options(
    optimization_level=1  # This is the default optimization level
)
pass_manager = generate_preset_pass_manager(
    backend=backend,
    initial_layout=list(gem_exp.physical_qubits),
    **dict(gem_exp.transpile_options),
)
transpiled = pass_manager.run(circuit)
transpiled.draw("mpl", idle_wires=False, fold=-1, scale=0.5)

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif" alt="Output of the previous code cell" />

Bagi tujuan tutorial ini, cukuplah untuk mempertimbangkan satu corak penjadualan sahaja. Sel kod berikut mengehadkan eksperimen kepada corak penjadualan pertama. Akibatnya, eksperimen hanya mempunyai 21 Circuit, satu untuk setiap sudut $R_{ZZ}$ yang disapu.

In [10]:
exp_data = gem_exp.run(shots=10_000)

To wait for the results, call the `block_for_results` method of the `ExperimentData` object. This call will cause the interpreter to hang until the jobs are finished.

In [11]:
exp_data.block_for_results()

ExperimentData(GemExperiment, d0d5880a-34c1-4aab-a7b6-c4f58516bc03, job_ids=['cwg12ptmptp00082khhg'], metadata=<5 items>, figure_names=['two_point_correlation.svg', 'normalized_variance.svg', 'plaquette_ops.svg', 'bond_ops.svg'])

Sel kod berikut melukis rajah Circuit pada indeks 5. Untuk mengurangkan saiz rajah, Gate pengukuran di penghujung Circuit dibuang.

In [ ]:
def magnetization_distribution(
    counts_dict: dict[str, int],
) -> dict[str, float]:
    """Compute magnetization distribution from counts dictionary."""
    # Construct dictionary from magnetization to count
    mag_dist = defaultdict(float)
    for bitstring, count in counts_dict.items():
        mag = bitstring.count("0") - bitstring.count("1")
        mag_dist[mag] += count
    # Normalize
    shots = sum(counts_dict.values())
    for mag in mag_dist:
        mag_dist[mag] /= shots
    return mag_dist


# Get counts dictionaries with and without decoding
data = exp_data.data()
# Get the last data point, which is at the angle for the GHZ state
raw_counts = data[-1]["counts"]
# Without decoding
site_indices = [
    i for i, q in enumerate(gem_exp.plaquettes.qubits()) if q.role == "Site"
]
site_raw_counts = defaultdict(int)
for key, val in raw_counts.items():
    site_str = "".join(key[-1 - i] for i in site_indices)
    site_raw_counts[site_str] += val
# With decoding
_, site_decoded_counts = gem_exp.plaquettes.decode_outcomes(
    raw_counts, return_counts=True
)

# Compute magnetization distribution
raw_magnetization = magnetization_distribution(site_raw_counts)
decoded_magnetization = magnetization_distribution(site_decoded_counts)

# Plot
plt.bar(*zip(*raw_magnetization.items()), label="raw")
plt.bar(*zip(*decoded_magnetization.items()), label="decoded", width=0.3)
plt.legend()
plt.xlabel("Magnetization")
plt.ylabel("Frequency")
plt.title("Magnetization distribution with and without decoding")

Text(0.5, 1.0, 'Magnetization distribution with and without decoding')

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/8ead3582-16df-4616-836c-bdce867ad6b8-1.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/fd57d483-c70b-4ad5-b309-15750ad38bac-0.avif)

## Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Transpilasi Circuit kuantum untuk pelaksanaan pada perkakasan biasanya melibatkan [beberapa peringkat](/guides/transpiler-stages). Biasanya, peringkat yang menanggung overhead pengiraan paling banyak adalah memilih susun atur Qubit, melaraskan Gate dua-Qubit agar sesuai dengan kesambungan Qubit perkakasan, dan mengoptimumkan Circuit untuk meminimumkan bilangan Gate dan kedalamannya. Dalam protokol GEM, peringkat susun atur dan penghalaan tidak diperlukan kerana kesambungan perkakasan telah dimasukkan ke dalam reka bentuk protokol. Circuit sudah mempunyai susun atur Qubit, dan Gate dua-Qubit sudah dipetakan ke sambungan natif. Tambahan pula, bagi memelihara struktur Circuit apabila sudut $R_{ZZ}$ berubah, hanya pengoptimuman Circuit yang sangat asas sahaja yang perlu dilakukan.

Kelas `GemExperiment` secara telus melakukan transpilasi Circuit ketika menjalankan eksperimen. Peringkat susun atur dan penghalaan sudah ditolak lalainya untuk tidak melakukan apa-apa, dan pengoptimuman Circuit dilakukan pada tahap yang hanya mengoptimumkan Gate satu-Qubit. Walau bagaimanapun, anda boleh menolak atau menghantar pilihan tambahan menggunakan kaedah `set_transpile_options`. Untuk tujuan visualisasi, sel kod berikut mentranspilkan Circuit yang dipaparkan sebelum ini secara manual, dan melukis Circuit yang telah ditranspilkan.

In [13]:
exp_data.figure("two_point_correlation")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/4ecb25c8-e572-49af-a879-9943039db131-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/e9b99d48-8d33-46b5-bff5-480ab1c1c1f2-0.avif)

## Langkah 3: Laksanakan menggunakan primitif Qiskit
Untuk melaksanakan Circuit protokol GEM pada perkakasan, panggil kaedah `run` objek `GemExperiment`. Anda boleh menentukan bilangan shot yang ingin anda sampel daripada setiap Circuit. Kaedah `run` mengembalikan objek [ExperimentData](https://qiskit-community.github.io/qiskit-experiments/stubs/qiskit_experiments.framework.ExperimentData.html) yang perlu anda simpan dalam pemboleh ubah. Perhatikan bahawa kaedah `run` hanya menghantar kerja tanpa menunggu ia selesai, jadi ini adalah panggilan bukan-penyekatan.

In [14]:
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/2b351d68-3924-445a-94ef-047b16214e8a-0.avif" alt="Output of the previous code cell" />

Untuk menunggu keputusan, panggil kaedah `block_for_results` objek `ExperimentData`. Panggilan ini akan menyebabkan penterjemah berhenti sehingga kerja-kerja selesai.

In [15]:
gem_exp = GemExperiment(
    plaquette_lattice.filter(range(3, 9)), backend=backend
)
gem_exp.set_experiment_options(schedule_idx=0)
exp_data = gem_exp.run(shots=10_000)
exp_data.block_for_results()
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/08581c09-a6a5-4a56-9fc4-abf22b063c6a-0.avif" alt="Output of the previous code cell" />

In [16]:
gem_exp = GemExperiment(plaquette_lattice, backend=backend)
gem_exp.set_experiment_options(schedule_idx=0)
exp_data = gem_exp.run(shots=10_000)
exp_data.block_for_results()
exp_data.figure("normalized_variance")

<Image src="../docs/images/tutorials/nishimori-phase-transition/extracted-outputs/37e9a4cd-6efb-4ade-ad09-8139db9d58e9-0.avif" alt="Output of the previous code cell" />

## Langkah 4: Pasca-proses dan kembalikan keputusan dalam format klasik yang dikehendaki
Pada sudut $R_{ZZ}$ sebesar $\frac{\pi}{2}$, keadaan yang dinyahkod akan menjadi keadaan GHZ tanpa kehadiran hingar. Perintah jarak jauh keadaan GHZ boleh divisualisasikan dengan memplot magnet rentetan bit yang diukur. Magnet $M$ ditakrifkan sebagai jumlah operator Pauli $Z$ satu-Qubit,
$$
M = \sum_{j=1}^N Z_j,
$$
di mana $N$ ialah bilangan Qubit tapak. Nilainya untuk satu rentetan bit adalah sama dengan perbezaan antara bilangan sifar dan bilangan satu. Mengukur keadaan GHZ menghasilkan keadaan semua sifar atau keadaan semua satu dengan kebarangkalian yang sama, jadi magnetnnya adalah $+N$ separuh masa dan $-N$ separuh masa lagi. Dengan kehadiran ralat akibat hingar, nilai-nilai lain juga akan muncul, tetapi jika hingar tidak terlalu besar, taburannya masih akan memuncak berhampiran $+N$ dan $-N$.

Bagi rentetan bit mentah sebelum penyahkodan, taburan magnet akan setara dengan rentetan bit rawak seragam, tanpa kehadiran hingar.

Sel kod berikut memplot magnet rentetan bit mentah dan rentetan bit yang dinyahkod pada sudut $R_{ZZ}$ sebesar $\frac{\pi}{2}$.